# CS801 Part 3 - Disaster Tweets: improved NLP classification and evaluation

This notebook is an improved, GitHub-ready version of the Disaster Tweets critique. It directly addresses feedback about over-reliance on accuracy by using stratified cross-validation, F1, precision, recall, ROC-AUC and PR-AUC, threshold tuning, error analysis, and a reproducible text-cleaning pipeline. The implementation favours lightweight scikit-learn tools so the notebook runs reliably on a clean GitHub environment.

In [ ]:
# MW-IMPROVED #24 - Imports and reproducibility
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import wilcoxon
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.cs801_utils import safe_read_csv, clean_tweet, binary_classification_summary

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 160)

In [ ]:
# MW-IMPROVED #25 - Data loading
DATA_DIR = PROJECT_ROOT / "data" / "raw"
train = safe_read_csv(DATA_DIR / "NLP_train.csv")
test_path = DATA_DIR / "NLP_test.csv"
test = safe_read_csv(test_path) if test_path.exists() else None

print(train.shape)
display(train.head())
display(train.isna().mean().mul(100).rename("missing_percent").to_frame())

In [ ]:
# MW-IMPROVED #26 - Class balance and evaluation implication
class_balance = train["target"].value_counts().sort_index().rename(index={0: "not_disaster", 1: "disaster"})
display(pd.DataFrame({"count": class_balance, "proportion": class_balance / len(train)}))

plt.figure(figsize=(6, 3))
sns.countplot(data=train, x="target")
plt.title("Disaster tweet class distribution")
plt.show()
print("Accuracy is reported but not used as the main success metric; F1, recall and PR-AUC are more informative for imbalanced alerting tasks.")

In [ ]:
# MW-IMPROVED #27 - Reproducible text cleaning and lightweight feature audit
train = train.copy()
train["clean_text"] = train["text"].map(clean_tweet)
if test is not None:
    test = test.copy()
    test["clean_text"] = test["text"].map(clean_tweet)

for df in [train] + ([] if test is None else [test]):
    df["char_count"] = df["clean_text"].str.len()
    df["word_count"] = df["clean_text"].str.split().map(len)
    df["hashtag_count"] = df["text"].astype(str).str.count("#")
    df["url_count"] = df["text"].astype(str).str.count(r"https?://|www\.")

display(train[["text", "clean_text", "target", "char_count", "word_count", "hashtag_count", "url_count"]].head())

plt.figure(figsize=(7, 4))
sns.kdeplot(data=train, x="char_count", hue="target", common_norm=False)
plt.title("Character-count distribution by class")
plt.show()

In [ ]:
# MW-IMPROVED #28 - Stratified holdout split plus stratified cross-validation
X = train["clean_text"]
y = train["target"].astype(int)

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Training class distribution:")
display(y_train.value_counts(normalize=True).sort_index())
print("Holdout class distribution:")
display(y_holdout.value_counts(normalize=True).sort_index())

In [ ]:
# MW-IMPROVED #29 - Model comparison with metrics beyond accuracy
# Stop words preserve negation words because "not/no/nor" can change tweet meaning.
custom_stop_words = sorted(set(ENGLISH_STOP_WORDS) - {"not", "no", "nor"})

models = {
    "tfidf_word_logreg_balanced": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=custom_stop_words, ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)),
    ]),
    "tfidf_char_logreg_balanced": Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)),
    ]),
    "tfidf_word_linear_svc": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=custom_stop_words, ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", LinearSVC(class_weight="balanced", C=1.0, random_state=RANDOM_STATE)),
    ]),
    "tfidf_word_multinomial_nb": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=custom_stop_words, ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", MultinomialNB(alpha=1.0)),
    ]),
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv_rows = []
fold_scores = {}
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    fold_scores[name] = scores["test_f1"]
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("f1_mean", ascending=False)
display(cv_results)

In [ ]:
# MW-IMPROVED #30 - Hyperparameter tuning optimised for F1 rather than accuracy
base_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words=custom_stop_words)),
    ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", solver="liblinear")),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.90, 0.95, 1.0],
    "clf__C": [0.3, 1.0, 3.0, 10.0],
}

grid = GridSearchCV(base_pipeline, param_grid=param_grid, scoring="f1", cv=cv, n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
print("Best F1 CV score:", grid.best_score_)
print("Best parameters:")
display(grid.best_params_)

In [ ]:
# MW-IMPROVED #31 - Holdout evaluation and threshold tuning
best_model = grid.best_estimator_
holdout_scores = best_model.predict_proba(X_holdout)[:, 1]
summary_05 = binary_classification_summary(y_holdout, holdout_scores, threshold=0.5)

precision, recall, thresholds = precision_recall_curve(y_holdout, holdout_scores)
f1_values = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
best_idx = int(np.nanargmax(f1_values[:-1])) if len(thresholds) else 0
best_threshold = float(thresholds[best_idx]) if len(thresholds) else 0.5
summary_best = binary_classification_summary(y_holdout, holdout_scores, threshold=best_threshold)

holdout_summary = pd.DataFrame([summary_05, summary_best], index=["threshold_0.50", "threshold_max_f1"])
display(holdout_summary)

holdout_pred = (holdout_scores >= best_threshold).astype(int)
print(classification_report(y_holdout, holdout_pred, digits=3))
display(pd.DataFrame(confusion_matrix(y_holdout, holdout_pred, labels=[0, 1]), index=["true_0", "true_1"], columns=["pred_0", "pred_1"]))

plt.figure(figsize=(7, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve for tuned TF-IDF Logistic Regression")
plt.show()

In [ ]:
# MW-IMPROVED #32 - Statistical comparison of fold F1 scores
# This is a small-sample paired test, so interpret it as supporting evidence rather than a definitive proof.
best_cv_model_name = cv_results.iloc[0]["model"]
baseline_name = "tfidf_word_multinomial_nb"
if best_cv_model_name in fold_scores and baseline_name in fold_scores and best_cv_model_name != baseline_name:
    stat, p_value = wilcoxon(fold_scores[best_cv_model_name], fold_scores[baseline_name], zero_method="wilcox", alternative="two-sided")
    display(pd.DataFrame([{
        "best_model": best_cv_model_name,
        "baseline_model": baseline_name,
        "wilcoxon_statistic": stat,
        "p_value": p_value,
        "best_f1_mean": np.mean(fold_scores[best_cv_model_name]),
        "baseline_f1_mean": np.mean(fold_scores[baseline_name]),
    }]))
else:
    print("Best CV model equals baseline or fold scores unavailable; skipping paired test.")

In [ ]:
# MW-IMPROVED #33 - Error analysis: false negatives and false positives
error_df = pd.DataFrame({
    "text": X_holdout.values,
    "target": y_holdout.values,
    "score_disaster": holdout_scores,
    "pred": holdout_pred,
})
false_negatives = error_df[(error_df["target"] == 1) & (error_df["pred"] == 0)].sort_values("score_disaster").head(10)
false_positives = error_df[(error_df["target"] == 0) & (error_df["pred"] == 1)].sort_values("score_disaster", ascending=False).head(10)

print("False negatives - real disasters missed by the model:")
display(false_negatives)
print("False positives - non-disasters predicted as disasters:")
display(false_positives)

In [ ]:
# MW-IMPROVED #34 - Optional test-set prediction file
# The file is created only when NLP_test.csv is present. Raw data and submissions are ignored by git.
if test is not None:
    test_scores = best_model.predict_proba(test["clean_text"])[:, 1]
    test_pred = (test_scores >= best_threshold).astype(int)
    submission = pd.DataFrame({"id": test["id"], "target": test_pred})
    output_path = PROJECT_ROOT / "outputs" / "disaster_tweets_submission.csv"
    output_path.parent.mkdir(exist_ok=True)
    submission.to_csv(output_path, index=False)
    print(f"Saved submission to {output_path}")
    display(submission.head())
else:
    print("NLP_test.csv not found; skipping submission file generation.")

## Improved conclusion for Part 3

The original notebook built a workable text classifier, but its evaluation leaned too much on accuracy and a single split. This version makes the quantitative evaluation stronger by preserving class ratios through stratification, comparing models with cross-validated F1/precision/recall/ROC-AUC/PR-AUC, tuning hyperparameters against F1, adjusting the decision threshold using the precision-recall curve, and inspecting false negatives and false positives. These changes make the notebook more defensible for both the CS801 assignment and a GitHub portfolio.